# Modelos de captioning multimodal para morfología celular

Notebook limpio para presentación/publicación. Incluye:

1. **BLIP + LoRA híbrido** como modelo principal.
2. **Gemma 3 VLM condicionado** como refinamiento multimodal opcional.
3. **Qwen-VL** como modelo multimodal opcional de comparación.

El código evita referencias a pacientes, hospitales o información identificable. Todas las entradas proceden de rutas de imagen y etiquetas del dataset.

## 1. INSTALACIÓN E IMPORTS

In [ ]:
!pip install -q transformers accelerate peft sentencepiece pillow pandas scikit-learn

import os
from pathlib import Path

import pandas as pd
import torch
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, PeftModel

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. RUTAS DEL PROYECTO

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# Directorio base del proyecto
BASE_DIR = Path(".").resolve()

RUTA_ANALISIS = BASE_DIR / "datos" / "analisis"
RUTA_ANALISIS.mkdir(parents=True, exist_ok=True)

# CSV completo con columnas:
# imagen, clase_real, instruction_blip, target_norm_blip_canonico
RUTA_DATASET_FULL = RUTA_ANALISIS / "dataset_blip_lora_hibrido_full.csv"
RUTA_TRAIN = RUTA_ANALISIS / "dataset_blip_lora_hibrido_train.csv"
RUTA_VAL = RUTA_ANALISIS / "dataset_blip_lora_hibrido_val.csv"

RUTA_SALIDA_MODELO = BASE_DIR / "modelos" / "blip_lora_hibrido_final"
RUTA_PREDICCIONES = RUTA_ANALISIS / "predicciones_blip_lora_hibrido.csv"

print("Directorio base:", BASE_DIR)
print("Ruta dataset:", RUTA_DATASET_FULL)

## 3. CONFIGURACIÓN DEL MODELO BLIP + LoRA HÍBRIDO

In [ ]:
PROCESSOR_NAME = "Salesforce/blip-image-captioning-base"
MODEL_NAME = "Salesforce/blip-image-captioning-base"

COLUMNA_IMAGEN = "imagen"
COLUMNA_CLASE = "clase_real"
COLUMNA_INPUT = "instruction_blip"
COLUMNA_TARGET = "target_norm_blip_canonico"

MAX_LENGTH_TEXT = 160

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

## 4. CARGA Y LIMPIEZA DEL DATASET

In [ ]:
def cargar_dataset():
    if RUTA_TRAIN.exists() and RUTA_VAL.exists():
        train_df = pd.read_csv(RUTA_TRAIN).fillna("")
        val_df = pd.read_csv(RUTA_VAL).fillna("")
        return train_df, val_df

    if not RUTA_DATASET_FULL.exists():
        raise FileNotFoundError(
            f"No se encuentra el dataset completo: {RUTA_DATASET_FULL}\n"
            "Crea ese CSV o ajusta RUTA_DATASET_FULL."
        )

    df = pd.read_csv(RUTA_DATASET_FULL).fillna("")

    columnas_necesarias = [COLUMNA_IMAGEN, COLUMNA_CLASE, COLUMNA_INPUT, COLUMNA_TARGET]
    faltantes = [c for c in columnas_necesarias if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas obligatorias: {faltantes}")

    for col in columnas_necesarias:
        df[col] = df[col].astype(str).str.strip()

    df = df[df[COLUMNA_IMAGEN] != ""]
    df = df[df[COLUMNA_CLASE] != ""]
    df = df[df[COLUMNA_INPUT] != ""]
    df = df[df[COLUMNA_TARGET] != ""]
    df = df[df[COLUMNA_IMAGEN].apply(os.path.exists)]
    df = df.drop_duplicates(subset=[COLUMNA_IMAGEN, COLUMNA_INPUT, COLUMNA_TARGET])
    df = df.reset_index(drop=True)

    conteos = df[COLUMNA_CLASE].value_counts()
    stratify_col = df[COLUMNA_CLASE] if not (conteos < 2).any() else None

    train_df, val_df = train_test_split(
        df,
        test_size=0.10,
        random_state=SEED,
        shuffle=True,
        stratify=stratify_col,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    train_df.to_csv(RUTA_TRAIN, index=False, encoding="utf-8")
    val_df.to_csv(RUTA_VAL, index=False, encoding="utf-8")

    return train_df, val_df


train_df, val_df = cargar_dataset()

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Clases:")
print(train_df[COLUMNA_CLASE].value_counts())

## 5. DATASET PARA BLIP + LoRA HÍBRIDO

In [ ]:
class BlipLoraHibridoDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.image_processor = processor.image_processor
        self.pad_token_id = self.tokenizer.pad_token_id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = str(row[COLUMNA_IMAGEN]).strip()
        prompt = str(row[COLUMNA_INPUT]).strip()
        target = str(row[COLUMNA_TARGET]).strip()

        image = Image.open(image_path).convert("RGB")
        full_text = f"{prompt} {target}".strip()

        image_inputs = self.image_processor(images=image)
        pixel_values = torch.tensor(
            image_inputs["pixel_values"][0],
            dtype=torch.float32,
        )

        text_inputs = self.tokenizer(
            full_text,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH_TEXT,
            return_tensors="pt",
        )

        prompt_inputs = self.tokenizer(
            prompt,
            padding=False,
            truncation=True,
            max_length=MAX_LENGTH_TEXT,
            return_tensors="pt",
        )

        input_ids = text_inputs["input_ids"].squeeze(0)
        attention_mask = text_inputs["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        labels[labels == self.pad_token_id] = -100

        prompt_len = prompt_inputs["input_ids"].shape[1]
        prompt_len = min(prompt_len, labels.shape[0])
        labels[:prompt_len] = -100

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

## 6. CARGA DEL MODELO Y CONFIGURACIÓN LoRA

In [ ]:
processor = BlipProcessor.from_pretrained(PROCESSOR_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.tie_word_embeddings = False


def detectar_modulos_lora_blip(model):
    """Selecciona módulos de atención del decodificador textual de BLIP."""
    modulos = []
    for name, _ in model.named_modules():
        if "text_decoder.bert.encoder.layer" not in name:
            continue
        if (
            name.endswith("attention.self.query")
            or name.endswith("attention.self.value")
            or name.endswith("crossattention.self.query")
            or name.endswith("crossattention.self.value")
        ):
            modulos.append(name)
    return sorted(set(modulos))


target_modules = detectar_modulos_lora_blip(model)
print("Módulos LoRA detectados:", len(target_modules))

if len(target_modules) == 0:
    raise ValueError("No se detectaron módulos compatibles con LoRA en BLIP.")

lora_config = LoraConfig(
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_dataset = BlipLoraHibridoDataset(train_df, processor)
val_dataset = BlipLoraHibridoDataset(val_df, processor)

## 7. ENTRENAMIENTO

In [ ]:
training_args = TrainingArguments(
    output_dir=str(RUTA_SALIDA_MODELO / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

RUTA_SALIDA_MODELO.mkdir(parents=True, exist_ok=True)
model.save_pretrained(RUTA_SALIDA_MODELO)
processor.save_pretrained(RUTA_SALIDA_MODELO)

print("Modelo BLIP + LoRA híbrido guardado en:", RUTA_SALIDA_MODELO)

## 8. INFERENCIA CON EL MODELO ENTRENADO

In [ ]:
def cargar_modelo_entrenado():
    processor_inf = BlipProcessor.from_pretrained(RUTA_SALIDA_MODELO)
    base_model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME)
    base_model.config.tie_word_embeddings = False
    model_inf = PeftModel.from_pretrained(base_model, RUTA_SALIDA_MODELO)
    model_inf = model_inf.to(DEVICE)
    model_inf.eval()
    return processor_inf, model_inf


processor_inf, model_inf = cargar_modelo_entrenado()


def generar_descripcion(image_path, prompt, max_new_tokens=80):
    image = Image.open(image_path).convert("RGB")

    inputs = processor_inf(
        images=image,
        text=prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH_TEXT,
    ).to(DEVICE)

    with torch.inference_mode():
        output_ids = model_inf.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            repetition_penalty=1.15,
            no_repeat_ngram_size=2,
            early_stopping=True,
        )

    generated_text = processor_inf.decode(output_ids[0], skip_special_tokens=True)
    generated_text = generated_text.replace(prompt, "").strip()
    return " ".join(generated_text.split())


# Generación sobre validación
predicciones = []

for _, row in val_df.iterrows():
    image_path = str(row[COLUMNA_IMAGEN]).strip()
    prompt = str(row[COLUMNA_INPUT]).strip()

    try:
        pred = generar_descripcion(image_path, prompt)
        error = ""
    except Exception as exc:
        pred = ""
        error = type(exc).__name__ + ": " + str(exc)

    predicciones.append({
        "imagen": image_path,
        "clase_real": row[COLUMNA_CLASE],
        "input_hibrido": prompt,
        "descripcion_referencia": row[COLUMNA_TARGET],
        "descripcion_generada_blip_lora_hibrido": pred,
        "error": error,
    })

pred_df = pd.DataFrame(predicciones)
pred_df.to_csv(RUTA_PREDICCIONES, index=False, encoding="utf-8")

print("Predicciones guardadas en:", RUTA_PREDICCIONES)
display(pred_df.head())

## 9. Preparación de entrada común para Gemma 3 y Qwen-VL

Esta parte usa como entrada el CSV de predicciones generado por BLIP + LoRA híbrido. Se conserva el lenguaje neutro: imagen, clase celular, descripción inicial y descripción de referencia.

In [ ]:
# --------------------------------------------------------------
# 9. PREPARACIÓN DE ENTRADA COMÚN PARA GEMMA 3 Y QWEN-VL
# -----------------------------------------------------------

RUTA_COMPARACION_ENTRADA = RUTA_PREDICCIONES
RUTA_GEMMA_OUT = RUTA_ANALISIS / "predicciones_gemma3_vlm_condicionado.csv"
RUTA_QWEN_OUT = RUTA_ANALISIS / "predicciones_qwen_vl.csv"

if not RUTA_COMPARACION_ENTRADA.exists():
    raise FileNotFoundError(
        f"No existe el CSV de entrada: {RUTA_COMPARACION_ENTRADA}. "
        "Ejecuta antes la inferencia de BLIP + LoRA híbrido."
    )

df_comp = pd.read_csv(RUTA_COMPARACION_ENTRADA).fillna("")

columnas_minimas = [
    "imagen",
    "clase_real",
    "input_hibrido",
    "descripcion_referencia",
    "descripcion_generada_blip_lora_hibrido",
]

faltantes = [c for c in columnas_minimas if c not in df_comp.columns]
if faltantes:
    raise ValueError(f"Faltan columnas necesarias: {faltantes}")

for col in columnas_minimas:
    df_comp[col] = df_comp[col].astype(str).str.strip()

df_comp = df_comp[df_comp["imagen"].apply(os.path.exists)].reset_index(drop=True)

print("Filas disponibles para comparación:", len(df_comp))
display(df_comp.head())

## 10. Gemma 3 VLM condicionado

Gemma 3 se usa aquí como refinador multimodal: recibe la imagen, la clase celular y la salida inicial de BLIP + LoRA híbrido. El objetivo es producir una frase morfológica más precisa sin introducir información externa.

In [ ]:
# -----------------------------------------------------
# 10. GEMMA 3 VLM CONDICIONADO
# -------------------------------------------------------
# Nota: este bloque requiere acceso al modelo en Hugging Face y suficiente memoria GPU.

!pip install -q --upgrade transformers accelerate huggingface_hub pillow

import re
import time
from tqdm.auto import tqdm
from transformers import pipeline

# Si el modelo requiere autenticación, descomenta estas líneas en Colab:
# from huggingface_hub import login
# from google.colab import userdata
# login(userdata.get("HF_TOKEN"))

MODEL_GEMMA = "google/gemma-3-4b-it"
MAX_NEW_TOKENS_GEMMA = 180
GUARDAR_CADA = 25
ESPERA_SEGUNDOS = 0.5


def construir_prompt_gemma(row):
    return f"""
You are a cell morphology captioning assistant.

Cell class: {row['clase_real']}
Reference description: {row['descripcion_referencia']}
Initial BLIP+LoRA hybrid caption: {row['descripcion_generada_blip_lora_hibrido']}

Task:
Improve the initial caption into one precise morphology sentence.
Use the image as the primary evidence.
Do not add unsupported findings.
Do not mention patients, hospitals, diagnosis, treatment, or personal data.
Return only the final caption.
""".strip()


def extraer_texto_pipeline(salida):
    gen = salida[0].get("generated_text", "")

    if isinstance(gen, list):
        ultimo = gen[-1]
        if isinstance(ultimo, dict):
            contenido = ultimo.get("content", "")
            if isinstance(contenido, list):
                return " ".join(
                    bloque.get("text", "")
                    for bloque in contenido
                    if isinstance(bloque, dict) and bloque.get("type") == "text"
                ).strip()
            return str(contenido).strip()
        return str(ultimo).strip()

    return str(gen).strip()


def limpiar_salida_modelo(texto):
    texto = str(texto).replace("```", "").strip()
    texto = re.sub(r"^\s*(assistant|model)\s*[:\-]\s*", "", texto, flags=re.I)
    texto = re.sub(r"\s+", " ", texto).strip(" \"'")
    if texto and texto[-1].isalnum():
        texto += "."
    return texto


pipe_gemma = pipeline(
    task="image-text-to-text",
    model=MODEL_GEMMA,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

# Reanudar si existe salida previa
if RUTA_GEMMA_OUT.exists():
    df_gemma = pd.read_csv(RUTA_GEMMA_OUT).fillna("")
    if len(df_gemma) != len(df_comp):
        df_gemma = df_comp.copy()
else:
    df_gemma = df_comp.copy()

for col in ["prediccion_gemma3", "prediccion_gemma3_raw", "error_gemma3"]:
    if col not in df_gemma.columns:
        df_gemma[col] = ""

procesadas = 0
errores = 0

for idx in tqdm(range(len(df_gemma)), desc="Generando con Gemma 3"):
    if str(df_gemma.at[idx, "prediccion_gemma3"]).strip():
        continue

    try:
        image = Image.open(df_gemma.at[idx, "imagen"]).convert("RGB")
        prompt = construir_prompt_gemma(df_gemma.iloc[idx])

        mensajes = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]

        salida = pipe_gemma(
            text=mensajes,
            max_new_tokens=MAX_NEW_TOKENS_GEMMA,
            do_sample=False,
        )

        raw = extraer_texto_pipeline(salida)
        final = limpiar_salida_modelo(raw)

        df_gemma.at[idx, "prediccion_gemma3_raw"] = raw
        df_gemma.at[idx, "prediccion_gemma3"] = final
        df_gemma.at[idx, "error_gemma3"] = ""
        procesadas += 1

    except Exception as exc:
        df_gemma.at[idx, "prediccion_gemma3_raw"] = ""
        df_gemma.at[idx, "prediccion_gemma3"] = ""
        df_gemma.at[idx, "error_gemma3"] = f"{type(exc).__name__}: {exc}"
        errores += 1

    if (idx + 1) % GUARDAR_CADA == 0:
        df_gemma.to_csv(RUTA_GEMMA_OUT, index=False, encoding="utf-8")

    time.sleep(ESPERA_SEGUNDOS)

df_gemma.to_csv(RUTA_GEMMA_OUT, index=False, encoding="utf-8")

print("Gemma 3 terminado")
print("Procesadas:", procesadas)
print("Errores:", errores)
print("Salida:", RUTA_GEMMA_OUT)
display(df_gemma.head())

## 11. Qwen-VL

Qwen-VL se usa como segundo modelo multimodal. El bloque mantiene la misma entrada que Gemma 3 para que las salidas sean comparables.

In [ ]:
# ------------------------------------------------------------
# 11. QWEN-VL
# -----------------------------------------------------
# Nota: este bloque requiere GPU y puede necesitar instalar transformers desde la versión más reciente.

!pip install -q --upgrade git+https://github.com/huggingface/transformers accelerate qwen-vl-utils pillow

import time
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

MODEL_QWEN = "Qwen/Qwen2.5-VL-3B-Instruct"
MAX_NEW_TOKENS_QWEN = 180
GUARDAR_CADA = 25
ESPERA_SEGUNDOS = 0.5


def construir_prompt_qwen(row):
    return f"""
You are a cell morphology captioning assistant.

Cell class: {row['clase_real']}
Reference description: {row['descripcion_referencia']}
Initial BLIP+LoRA hybrid caption: {row['descripcion_generada_blip_lora_hibrido']}

Task:
Generate one precise morphology sentence.
Use the image as the primary evidence.
Do not add unsupported findings.
Do not mention patients, hospitals, diagnosis, treatment, or personal data.
Return only the final caption.
""".strip()


processor_qwen = AutoProcessor.from_pretrained(MODEL_QWEN, trust_remote_code=True)
model_qwen = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_QWEN,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
model_qwen.eval()

# Reanudar si existe salida previa
if RUTA_QWEN_OUT.exists():
    df_qwen = pd.read_csv(RUTA_QWEN_OUT).fillna("")
    if len(df_qwen) != len(df_comp):
        df_qwen = df_comp.copy()
else:
    df_qwen = df_comp.copy()

for col in ["prediccion_qwen_vl", "prediccion_qwen_vl_raw", "error_qwen_vl"]:
    if col not in df_qwen.columns:
        df_qwen[col] = ""

procesadas = 0
errores = 0

for idx in tqdm(range(len(df_qwen)), desc="Generando con Qwen-VL"):
    if str(df_qwen.at[idx, "prediccion_qwen_vl"]).strip():
        continue

    try:
        image_path = str(df_qwen.at[idx, "imagen"]).strip()
        prompt = construir_prompt_qwen(df_qwen.iloc[idx])

        mensajes = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ],
        }]

        text = processor_qwen.apply_chat_template(
            mensajes,
            tokenize=False,
            add_generation_prompt=True,
        )
        image_inputs, video_inputs = process_vision_info(mensajes)

        inputs = processor_qwen(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(model_qwen.device)

        with torch.inference_mode():
            generated_ids = model_qwen.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS_QWEN,
                do_sample=False,
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        raw = processor_qwen.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]

        final = limpiar_salida_modelo(raw)

        df_qwen.at[idx, "prediccion_qwen_vl_raw"] = raw
        df_qwen.at[idx, "prediccion_qwen_vl"] = final
        df_qwen.at[idx, "error_qwen_vl"] = ""
        procesadas += 1

    except Exception as exc:
        df_qwen.at[idx, "prediccion_qwen_vl_raw"] = ""
        df_qwen.at[idx, "prediccion_qwen_vl"] = ""
        df_qwen.at[idx, "error_qwen_vl"] = f"{type(exc).__name__}: {exc}"
        errores += 1

    if (idx + 1) % GUARDAR_CADA == 0:
        df_qwen.to_csv(RUTA_QWEN_OUT, index=False, encoding="utf-8")

    time.sleep(ESPERA_SEGUNDOS)

df_qwen.to_csv(RUTA_QWEN_OUT, index=False, encoding="utf-8")

print("Qwen-VL terminado")
print("Procesadas:", procesadas)
print("Errores:", errores)
print("Salida:", RUTA_QWEN_OUT)
display(df_qwen.head())

## 12. Tabla final de comparación de salidas

Esta tabla junta las descripciones de BLIP + LoRA híbrido, Gemma 3 y Qwen-VL. No calcula métricas automáticamente; sirve para revisión cualitativa o para una evaluación posterior.

In [ ]:
# --------------------------------------------------------
# 12. TABLA FINAL DE COMPARACIÓN DE SALIDAS
# ---------------------------------------------------------

RUTA_COMPARACION_FINAL = RUTA_ANALISIS / "comparacion_blip_lora_gemma3_qwen_vl.csv"

comparacion = df_comp[[
    "imagen",
    "clase_real",
    "descripcion_referencia",
    "descripcion_generada_blip_lora_hibrido",
]].copy()

if RUTA_GEMMA_OUT.exists():
    gemma_tmp = pd.read_csv(RUTA_GEMMA_OUT).fillna("")
    if "prediccion_gemma3" in gemma_tmp.columns:
        comparacion["prediccion_gemma3"] = gemma_tmp["prediccion_gemma3"].astype(str)

if RUTA_QWEN_OUT.exists():
    qwen_tmp = pd.read_csv(RUTA_QWEN_OUT).fillna("")
    if "prediccion_qwen_vl" in qwen_tmp.columns:
        comparacion["prediccion_qwen_vl"] = qwen_tmp["prediccion_qwen_vl"].astype(str)

comparacion.to_csv(RUTA_COMPARACION_FINAL, index=False, encoding="utf-8")

print("Comparación final guardada en:", RUTA_COMPARACION_FINAL)
display(comparacion.head())